In [1]:
import mne
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.signal import welch, hilbert, detrend
import utils
import torch
from torch.utils.data import Dataset

In [2]:
import mne
import pandas as pd

blocks = []
sfreq = None

for block_id in range(1, 13):
    edf_file = f"data/sub-001_ses-04_block-{block_id:03d}_task-ssvep_eeg.edf"
    tsv_file = f"data/sub-001_ses-04_block-{block_id:03d}_task-ssvep_events.tsv"

    data = mne.io.read_raw_edf(edf_file, preload=True)
    raw = data.get_data()
    events_df = pd.read_csv(tsv_file, sep="\t")

    if sfreq is None:
        sfreq = data.info["sfreq"]

    # Select 12 Hz events (adjust column name if needed)
    stim_12hz = events_df[(events_df["stim_frequency"] == 12) & (events_df["value"] % 2 == 0)]

    # Extract all 12 Hz segments in this block
    segments = []
    for _, row in stim_12hz.iterrows():
        onset = row["onset"]          # seconds
        duration_sec = 5
        duration_samples = (duration_sec * sfreq)+124

        start = onset
        end   = onset + duration_samples # 5 s + 125 ms

        seg = data.copy().crop(
            tmin=start / sfreq,
            tmax=end   / sfreq
            )
        segments.append(seg)

    # Concatenate segments *within* block
    if len(segments) > 0:
        block_raw = mne.concatenate_raws(segments)
        blocks.append(block_raw)


Extracting EDF parameters from c:\Users\danie_13ucdo4\OneDrive\Desktop\ITAM\Tesis\Prueba\timegan2.0\data\sub-001_ses-04_block-001_task-ssvep_eeg.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 525999  =      0.000 ...   525.999 secs...
Extracting EDF parameters from c:\Users\danie_13ucdo4\OneDrive\Desktop\ITAM\Tesis\Prueba\timegan2.0\data\sub-001_ses-04_block-002_task-ssvep_eeg.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 525999  =      0.000 ...   525.999 secs...
Extracting EDF parameters from c:\Users\danie_13ucdo4\OneDrive\Desktop\ITAM\Tesis\Prueba\timegan2.0\data\sub-001_ses-04_block-003_task-ssvep_eeg.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 526999  =      0.000 ...   526.999 secs...
Extracting EDF parameters from c:\Users\danie_13ucdo4\OneDrive\Desktop\ITAM\Tesis\Prueba\timegan2.0\data\sub-001_ses-04_bloc

In [3]:
blocks

[<RawEDF | sub-001_ses-04_block-001_task-ssvep_eeg.edf, 64 x 5125 (5.1 s), ~2.6 MB, data loaded>,
 <RawEDF | sub-001_ses-04_block-002_task-ssvep_eeg.edf, 64 x 5125 (5.1 s), ~2.6 MB, data loaded>,
 <RawEDF | sub-001_ses-04_block-003_task-ssvep_eeg.edf, 64 x 5125 (5.1 s), ~2.6 MB, data loaded>,
 <RawEDF | sub-001_ses-04_block-004_task-ssvep_eeg.edf, 64 x 5125 (5.1 s), ~2.6 MB, data loaded>,
 <RawEDF | sub-001_ses-04_block-005_task-ssvep_eeg.edf, 64 x 5125 (5.1 s), ~2.6 MB, data loaded>,
 <RawEDF | sub-001_ses-04_block-006_task-ssvep_eeg.edf, 64 x 5125 (5.1 s), ~2.6 MB, data loaded>,
 <RawEDF | sub-001_ses-04_block-007_task-ssvep_eeg.edf, 64 x 5125 (5.1 s), ~2.6 MB, data loaded>,
 <RawEDF | sub-001_ses-04_block-008_task-ssvep_eeg.edf, 64 x 5125 (5.1 s), ~2.6 MB, data loaded>,
 <RawEDF | sub-001_ses-04_block-009_task-ssvep_eeg.edf, 64 x 5125 (5.1 s), ~2.6 MB, data loaded>,
 <RawEDF | sub-001_ses-04_block-010_task-ssvep_eeg.edf, 64 x 5125 (5.1 s), ~2.6 MB, data loaded>,
 <RawEDF | sub-001_s

In [4]:
wanted_channels = ['PZ', 'PO3', 'PO4', 'PO5', 'PO6', 'POZ', 'OZ', 'O1', 'O2']

In [5]:
all_segments = []

for block_raw in blocks:
    block_raw.filter(
    l_freq=10,
    h_freq=40,
    fir_design="firwin",
    phase="zero"
    )
    data = block_raw.get_data(picks=wanted_channels)   # shape: (n_channels, n_samples)
    all_segments.append(data)

eeg = np.concatenate(all_segments, axis=1)


Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 10 - 40 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 10.00
- Lower transition bandwidth: 2.50 Hz (-6 dB cutoff frequency: 8.75 Hz)
- Upper passband edge: 40.00 Hz
- Upper transition bandwidth: 10.00 Hz (-6 dB cutoff frequency: 45.00 Hz)
- Filter length: 1321 samples (1.321 s)

Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 10 - 40 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 10.00
- Lower transition bandwidth: 2.50 Hz (-6 dB cutoff frequency: 8.75 Hz)
- Upper passband ed

[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.0s


Setting up band-pass filter from 10 - 40 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 10.00
- Lower transition bandwidth: 2.50 Hz (-6 dB cutoff frequency: 8.75 Hz)
- Upper passband edge: 40.00 Hz
- Upper transition bandwidth: 10.00 Hz (-6 dB cutoff frequency: 45.00 Hz)
- Filter length: 1321 samples (1.321 s)

Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 10 - 40 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 10.00
- Lower transition bandwidth: 2.50 Hz (-6 dB cutoff frequency: 8.75 Hz)
- Upper passband edge: 40.00 Hz
- Upper transition bandwidth: 

[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.0s


In [6]:
eeg = eeg.T
eeg.shape

(61500, 9)

In [7]:
class EEGWindowDataset(Dataset):
    """
    Windowed EEG Dataset for TimeGAN.

    Returns:
        X: torch.Tensor of shape [T, C]
    """

    def __init__(
        self,
        eeg,
        window_size,
        hop_size,
        normalize=True,
        stats=None,
        dtype=torch.float32
    ):
        """
        Parameters
        ----------
        eeg : np.ndarray
            Shape either:
              [T_total, C] or
              [N_trials, T_trial, C]

        window_size : int
            Number of time samples per window (e.g., 2000)

        hop_size : int
            Hop size between windows (e.g., 1000)

        normalize : bool
            Whether to apply per-channel normalization

        stats : dict or None
            Precomputed normalization stats:
              {'mean': np.ndarray[C], 'std': np.ndarray[C]}
            If None and normalize=True, stats are computed internally.

        dtype : torch.dtype
            Output tensor dtype
        """

        self.window_size = window_size
        self.hop_size = hop_size
        self.normalize = normalize
        self.dtype = dtype

        # Ensure 3D: [N_trials, T, C]
        if eeg.ndim == 2:
            eeg = eeg[None, ...]
        elif eeg.ndim != 3:
            raise ValueError("EEG must have shape [T, C] or [N, T, C]")

        self.eeg = eeg
        self.n_trials, self.T, self.C = eeg.shape

        # Precompute window indices
        self.index_map = []
        for trial in range(self.n_trials):
            max_start = self.T - window_size
            for start in range(0, max_start + 1, hop_size):
                self.index_map.append((trial, start))

        # Normalization
        if self.normalize:
            if stats is None:
                self.mean, self.std = self._compute_stats()
            else:
                self.mean = stats["mean"]
                self.std = stats["std"]

            # Numerical safety
            self.std = np.maximum(self.std, 1e-6)

    def _compute_stats(self):
        """
        Compute per-channel mean and std over ALL trials and time.
        """
        data = self.eeg.reshape(-1, self.C)
        mean = data.mean(axis=0)
        std = data.std(axis=0)
        return mean, std

    def __len__(self):
        return len(self.index_map)

    def __getitem__(self, idx):
        trial, start = self.index_map[idx]
        window = self.eeg[
            trial,
            start : start + self.window_size,
            :
        ]

        if self.normalize:
            window = (window - self.mean) / self.std

        return torch.tensor(window, dtype=self.dtype)

In [17]:
window_size = 1000   # 2 seconds at 1000 Hz
hop_size = 250     # 50% overlap

dataset = EEGWindowDataset(
    eeg=eeg,    # np.ndarray
    window_size=window_size,
    hop_size=hop_size,
    normalize=True
)


In [18]:
from torch.utils.data import DataLoader

dataloader = DataLoader(
    dataset,
    batch_size=8,
    shuffle=True,
    drop_last=False
)

In [19]:
print("Number of windows:", len(dataset))

Number of windows: 243
